### Notebook scope

### Purpose
Compare onset timing of EP, vortex wind, and O3 ensemble spread.

### Scientific question
Does dynamical spread emerge before ozone spread across cases?

### Method
Spread is ensemble standard deviation; onset is first 5-day running mean exceeding 50 percent of its own maximum for at least 5 days.

### Expected output
One timeline figure and one CSV onset summary.

### Interpretation
EP/U onset before O3 onset supports dynamics-first uncertainty.

### Caveat
Later-initialized cases may have dynamical divergence already embedded at initialization.


### Setup imports and shared paths

### Purpose
Load the shared Extention_analysis utility module and create output directories.

### Scientific question
Can every notebook start from a clean kernel and still find the same data roots and output roots?

### Method
Use DATA_ROOT, HINDCAST_ROOT, BWCN_ROOT, B2000_ROOT, WORK_ROOT, FIG_DIR, TAB_DIR, CACHE_DIR, and LOG_DIR from hindcast_extension_utils.py. No diagnostics are calculated in this cell.

### Expected output
Printed path check only. No figure is generated by this setup cell.

### Interpretation
Successful execution means the notebook can access the common utilities and write outputs in the agreed directory tree.

### Caveat
This setup does not prove that every downstream data product exists; missing products are logged later.


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

import hindcast_extension_utils as hx
from hindcast_extension_utils import *
_assign_member_short = hx._assign_member_short

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)

### Compute spread-onset timing

### Purpose
Compute spread onset for O3, EP100 W1+W2, U60N10, and U60N50 for available cases.

### Scientific question
Is the ordering EP/U before O3 repeated outside 0008-01?

### Method
O3 uses partial column; EP100 W1+W2 is wave1+wave2 at 100 hPa, 40-80N; U winds are 60N at 10/50 hPa.

### Expected output
outputs/tables/04_spread_onset_timing_summary_all_cases.csv and outputs/figures/04_spread_onset_timing_summary_all_cases.png/pdf.

### Interpretation
Markers to the left indicate earlier spread onset. Dynamics-first cases should show EP/U markers before O3 markers.

### Caveat
Spread onset is threshold-based and sensitive to small ensemble size and noisy cases.


In [ ]:
inv = discover_hindcast_cases()
rows = []
for case in inv.loc[inv["has_partial_O3"], "case"]:
    meta = parse_case_name(case)
    o3, date = load_hindcast_o3(case)
    if o3 is not None:
        onset = compute_spread_onset(o3)
        rows.append({"case": case, "variable": "O3", **{k:v for k,v in onset.items() if not isinstance(v, np.ndarray)}})
    ep1, epdate = load_epflux(case, "wave1")
    ep2, _ = load_epflux(case, "wave2")
    if ep1 is not None and ep2 is not None:
        ep = coslat_weighted_mean(-(ep1.sel(plev=10000, method="nearest") + ep2.sel(plev=10000, method="nearest")), LAT_EP)
        onset = compute_spread_onset(ep)
        rows.append({"case": case, "variable": "EP100_W1W2", **{k:v for k,v in onset.items() if not isinstance(v, np.ndarray)}})
    for plev in [10, 50]:
        u, udate = compute_u60(case, plev)
        if u is not None:
            onset = compute_spread_onset(u)
            rows.append({"case": case, "variable": f"U60N{plev}", **{k:v for k,v in onset.items() if not isinstance(v, np.ndarray)}})
spread = pd.DataFrame(rows)
spread_csv = TAB_DIR / "04_spread_onset_timing_summary_all_cases.csv"
spread.to_csv(spread_csv, index=False)
fig, ax = plt.subplots(figsize=(11, max(5, 0.36*spread["case"].nunique()+2 if not spread.empty else 5)), constrained_layout=True)
if spread.empty:
    ax.axis("off"); ax.text(0.5, 0.5, "No spread onset diagnostics", ha="center", va="center")
else:
    cases = sorted(spread["case"].unique())
    ymap = {c:i for i,c in enumerate(cases)}
    markers = {"EP100_W1W2":"^", "U60N10":"o", "U60N50":"s", "O3":"D"}
    colors = {"EP100_W1W2":"tab:purple", "U60N10":"tab:orange", "U60N50":"tab:green", "O3":"tab:blue"}
    for var, sub in spread.groupby("variable"):
        ax.scatter(sub["onset_doy"], [ymap[c] for c in sub["case"]], marker=markers.get(var, "o"), s=80, color=colors.get(var), label=var, edgecolor="black")
    ax.set_yticks(range(len(cases)), cases)
    ax.set_xlabel("Onset day of year")
    ax.set_title("Spread onset timing summary")
    ax.legend(ncol=4, fontsize=9)
savefig(fig, "04_spread_onset_timing_summary_all_cases", notebook="04_spread_timing_multicase.ipynb", scientific_question="Does EP/vortex spread emerge before O3 spread?", variables_windows="O3 partial column; EP100 W1+W2; U60N10; U60N50; Jan-May available lead", interpretation="EP/U markers earlier than O3 markers support dynamics-first uncertainty propagation.", caveat="Threshold onset can be ambiguous when spread evolves gradually or data begin after divergence has already started.", csv_table=spread_csv)
plt.show()
write_figure_guide()